In [3]:
import gzip
import json
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import glob

In [4]:
def load_json_gzp(path):
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

def add_geo_cells(df, cell_size=0.1):
    d = df.copy()
    # Create grid cells based on lat/long
    d["lat_cell"] = (d["latitude"] // cell_size) * cell_size
    d["lon_cell"] = (d["longitude"] // cell_size) * cell_size
    d["geo_cell"] = d["lat_cell"].astype(str) + "_" + d["lon_cell"].astype(str)
    return d

def process_state_volatility(meta_path):
    state_name = meta_path.split('-')[-1].split('.')[0]
    print(f"Processing {state_name}...")
    

    df_meta = load_json_gzp(meta_path)
    df_meta = df_meta[["gmap_id", "avg_rating", "num_of_reviews", "latitude", "longitude"]].dropna()
    df_meta["num_of_reviews"] = pd.to_numeric(df_meta["num_of_reviews"])
    
    # 2. Derive Urban/Rural Regions
    df_meta = add_geo_cells(df_meta)
    
    cell_features = df_meta.groupby("geo_cell").agg(
        n_businesses=("gmap_id", "nunique"),
        avg_reviews_per_biz=("num_of_reviews", "mean"),
        pct_high_review_biz=("num_of_reviews", lambda x: (x >= 100).mean())
    )
    
    # Create the Urban Index using Scaling
    scaler = StandardScaler()
    indexed_features = scaler.fit_transform(cell_features)
    cell_features["urban_index"] = indexed_features.mean(axis=1)
    
    # Split into Urban/Rural (Top 50% density = Urban)
    cell_features["urban_rural"] = pd.qcut(cell_features["urban_index"], q=2, labels=["rural", "urban"])
    
    # Merge label back to businesses
    df_meta = df_meta.merge(cell_features[["urban_rural"]], on="geo_cell", how="left")
    
    # Create Volatility Bins
    bins = [1, 5, 10, 20, 50, 100, 200, 500, 1000, 5001]
    labels = ["1–4", "5–9", "10–19", "20–49", "50–99", "100–199", "200–499", "500–999", "1000+"]
    df_meta["review_bin"] = pd.cut(df_meta["num_of_reviews"], bins=bins, labels=labels, right=False)
    
    # STATEWIDE 
    state_summary = df_meta.groupby("review_bin").agg(
        n_businesses=("gmap_id", "count"),
        rating_std=("avg_rating", "std"),
        mean_rating=("avg_rating", "mean")
    ).reset_index()
    state_summary["state"] = state_name
    
    # REGIONAL (Urban vs Rural)
    regional_summary = df_meta.groupby(["urban_rural", "review_bin"]).agg(
        n_businesses=("gmap_id", "count"),
        rating_std=("avg_rating", "std"),
        mean_rating=("avg_rating", "mean")
    ).reset_index()
    regional_summary["state"] = state_name
    
    return state_summary, regional_summary

# RUN FOR ALL STATES IN DATA FOLDER
all_state_files = glob.glob("data/meta-*.json.gz")
state_results = []
regional_results = []

for file in all_state_files:
    s_sum, r_sum = process_state_volatility(file)
    state_results.append(s_sum)
    regional_results.append(r_sum)

# Export Master Files for Tableau
pd.concat(state_results).to_csv("tableau_state_volatility.csv", index=False)
pd.concat(regional_results).to_csv("tableau_regional_volatility.csv", index=False)

Processing District_of_Columbia...


C:\Users\aiden\AppData\Local\Temp\ipykernel_13616\2621918889.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  state_summary = df_meta.groupby("review_bin").agg(
C:\Users\aiden\AppData\Local\Temp\ipykernel_13616\2621918889.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["urban_rural", "review_bin"]).agg(


Processing Virginia...


C:\Users\aiden\AppData\Local\Temp\ipykernel_13616\2621918889.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  state_summary = df_meta.groupby("review_bin").agg(
C:\Users\aiden\AppData\Local\Temp\ipykernel_13616\2621918889.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["urban_rural", "review_bin"]).agg(


Processing Wyoming...


C:\Users\aiden\AppData\Local\Temp\ipykernel_13616\2621918889.py:51: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  state_summary = df_meta.groupby("review_bin").agg(
C:\Users\aiden\AppData\Local\Temp\ipykernel_13616\2621918889.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  regional_summary = df_meta.groupby(["urban_rural", "review_bin"]).agg(
